In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Load dataset
data = pd.read_csv('/content/content_refresh_anonymized.csv')

print("Dataset Shape:", data.shape)

Dataset Shape: (30000, 44)


In [3]:
# Fill missing values with median
data['scroll_rate'] = data['scroll_rate'].fillna(
    data['scroll_rate'].median()
)

data['trend_pct'] = data['trend_pct'].fillna(
    data['trend_pct'].median()
)

print("Missing values after cleaning:")
print(
    data[['scroll_rate', 'trend_pct']]
        .isnull()
            .sum()
)

Missing values after cleaning:
scroll_rate    0
trend_pct      0
dtype: int64


In [4]:
# Metrics used for baseline scoring
score_data = data[
    [
        'ctr',
        'engagement_rate',
        'scroll_rate',
        'avg_position',
        'trend_pct'
    ]
].copy()

# Normalize metrics to 0–1
scaler = MinMaxScaler()

score_data[
    [
        'ctr',
        'engagement_rate',
        'scroll_rate',
        'avg_position',
        'trend_pct'
    ]
] = scaler.fit_transform(
    score_data[
        [
            'ctr',
            'engagement_rate',
            'scroll_rate',
            'avg_position',
            'trend_pct'
        ]
    ]
)

# Lower search position is better
score_data['position_score'] = 1 - score_data['avg_position']

# Calculate baseline score
data['baseline_score'] = (
    0.25 * score_data['ctr'] +
    0.20 * score_data['engagement_rate'] +
    0.15 * score_data['scroll_rate'] +
    0.25 * score_data['position_score'] +
    0.15 * score_data['trend_pct']
)

print(data[['content_id', 'baseline_score']].head())

             content_id  baseline_score
0  content_304f48230142        0.255314
1  content_a1fb4e703a9e        0.234552
2  content_9aa793d4d895        0.227395
3  content_331d6c4de07b        0.249471
4  content_d99b7a2d90ca        0.217790


In [5]:
# Rank pages by baseline score
data['baseline_rank'] = data['baseline_score'].rank(
    ascending=False,
    method='min'
)

ranked_data = data.sort_values(
    'baseline_score',
    ascending=False
)

display(
    ranked_data[
        ['content_id', 'baseline_score', 'baseline_rank']
    ].head(10)
)

,content_id,baseline_score,baseline_rank
240,content_006b16e7a2e7,0.724201,1.0
19341,content_4272d3a330a3,0.704558,2.0
23549,content_78467f5a1702,0.625222,3.0
6565,content_7debdb0d3a8a,0.624201,4.0
1190,content_e1f00d5f0d0c,0.592058,5.0
21901,content_76b07f20b83c,0.587722,6.0
903,content_2278cd2eba78,0.582832,7.0
10083,content_d28d84af56c2,0.576404,8.0
1239,content_2b5f1b817381,0.559754,9.0
9656,content_397c3fd4ec98,0.555783,10.0


In [6]:
# Rank pages by baseline score
data['baseline_rank'] = data['baseline_score'].rank(
    ascending=False,
    method='min'
)

# Sort pages from highest to lowest score
ranked_data = data.sort_values(
    'baseline_score',
    ascending=False
)

# Display top 10 pages
display(
    ranked_data[
        ['content_id', 'baseline_score', 'baseline_rank']
    ].head(10)
)

,content_id,baseline_score,baseline_rank
240,content_006b16e7a2e7,0.724201,1.0
19341,content_4272d3a330a3,0.704558,2.0
23549,content_78467f5a1702,0.625222,3.0
6565,content_7debdb0d3a8a,0.624201,4.0
1190,content_e1f00d5f0d0c,0.592058,5.0
21901,content_76b07f20b83c,0.587722,6.0
903,content_2278cd2eba78,0.582832,7.0
10083,content_d28d84af56c2,0.576404,8.0
1239,content_2b5f1b817381,0.559754,9.0
9656,content_397c3fd4ec98,0.555783,10.0


In [7]:
# Save the baseline-scored dataset
output_file = '/content/flyrank_baseline_scored.csv'

data.to_csv(output_file, index=False)

print("Baseline CSV saved successfully!")
print("File:", output_file)
print("Shape:", data.shape)

Baseline CSV saved successfully!
File: /content/flyrank_baseline_scored.csv
Shape: (30000, 46)
